<a href="https://colab.research.google.com/github/shashikant-ds636/easycluster/blob/main/examples/clustereasy_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# clustereasy — Quickstart

Run this notebook directly in Google Colab, Jupyter, or VS Code. It installs
`clustereasy`, builds a synthetic customer dataset, and walks through **every**
method on the `Cluster` class: K-Means, Agglomerative Clustering, evaluation
metrics, plots, and the DataFrame-output helpers.

**v0.1.3 note:** every `plot_*` method and `elbow_method()` now return
`None` — they only draw the plot, so a notebook cell shows just the chart,
not a second printed table/array underneath it. Use the matching getter
(`elbow_results()`, `get_linkage_matrix()`, etc.) when you want the data
back. `fit_kmeans()` / `fit_agglomerative()` still return `self`, so fit
calls remain chainable.

In [ ]:
# Works the same in Colab, Jupyter, and plain scripts.
# In Colab/Jupyter, the leading ! runs it as a shell command.
!pip install -q clustereasy

In [ ]:
import numpy as np
import pandas as pd
from clustereasy import Cluster

np.random.seed(42)
n = 200
df = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Annual Income": np.concatenate([np.random.normal(25, 5, n // 2), np.random.normal(75, 8, n - n // 2)]),
    "Spending Score": np.concatenate(
        [np.random.normal(30, 10, n // 2), np.random.normal(70, 12, n - n // 2)]
    ),
})
df.head()

## 1. Build the `Cluster` object

Wraps a DataFrame, picks feature columns, and optionally scales them.

In [ ]:
cluster = Cluster(
    data=df,
    features=["Annual Income", "Spending Score"],  # omit to use all columns
    scale=True,       # True == "standard"; also accepts "minmax" / "robust"
    missing="raise",  # or "drop" / "mean" / "median"
)
cluster

## 2. K-Means

### `elbow_method()` + `elbow_results()`

`elbow_method()` fits K-Means for k = 1..max_clusters and **only plots** WCSS
vs. k (with a suggested-k marker as a visual hint). Call `elbow_results()`
afterwards to get the numbers back as a DataFrame.

In [ ]:
cluster.elbow_method(max_clusters=8, random_state=42)  # plot only — returns None

In [ ]:
cluster.elbow_results()  # the k/WCSS table, fetched separately

### `fit_kmeans()`

Returns `self`, so it's chainable. Explicit, reproducible defaults.

In [ ]:
cluster.fit_kmeans(
    n_clusters=4,
    init="k-means++",
    n_init=10,
    max_iter=300,
    tol=1e-4,
    random_state=42,
    algorithm="lloyd",
)

### `cluster_centers()` and `inertia()`

K-Means-only. `cluster_centers()` defaults to original feature scale.

In [ ]:
cluster.cluster_centers()

In [ ]:
cluster.cluster_centers(original_scale=False)  # in the scaled space the model actually used

In [ ]:
cluster.inertia()  # WCSS for the fitted model

### `predict()`

Assign *new* rows to the existing K-Means clusters. K-Means only —
Agglomerative has no native `predict()` in scikit-learn.

In [ ]:
new_customers = pd.DataFrame({
    "Annual Income": [24, 78, 50],
    "Spending Score": [28, 72, 50],
})
cluster.predict(new_customers)

## 3. Shared plots (work after fitting either algorithm)

### `plot_clusters()`

2D scatter of the fitted clusters. Plot only — returns `None`.

In [ ]:
cluster.plot_clusters(x="Annual Income", y="Spending Score", show_centers=True)

### `plot_cluster_sizes()`

Bar chart of how many points landed in each cluster.

In [ ]:
cluster.plot_cluster_sizes()

### `plot_silhouette()`

Per-sample silhouette diagram, one band per cluster, with the average marked.

In [ ]:
cluster.plot_silhouette()

### `plot_pca()`

2D PCA projection — useful when you clustered on more than 2 features.

In [ ]:
cluster.plot_pca()

## 4. Evaluation metrics (no ground-truth labels needed)

In [ ]:
cluster.silhouette_score()

In [ ]:
cluster.silhouette_samples().head()  # per-sample silhouette values, indexed like the data

In [ ]:
cluster.davies_bouldin_score()  # lower is better

In [ ]:
cluster.calinski_harabasz_score()  # higher is better

In [ ]:
cluster.evaluate()  # all applicable metrics (+ inertia for K-Means) in one table

## 5. Results / DataFrame integration

In [ ]:
cluster.get_labels().head()  # cluster id per row, as a Series

In [ ]:
cluster.get_clustered_data(column_name="Segment").head()  # original data + a cluster-label column

In [ ]:
cluster.cluster_summary(agg="mean")  # size, proportion, and per-feature mean by cluster

## 6. Agglomerative (Hierarchical) Clustering

Uses a fresh `Cluster` object — a single instance holds at most one fitted
model at a time, and K-Means already fitted the one above.

In [ ]:
cluster2 = Cluster(data=df, features=["Annual Income", "Spending Score"], scale=True)

### `plot_dendrogram()` + `get_linkage_matrix()`

Plot only — returns `None`. Call `get_linkage_matrix()` afterwards for the
raw SciPy linkage matrix.

In [ ]:
cluster2.plot_dendrogram(method="ward", metric="euclidean")

In [ ]:
cluster2.get_linkage_matrix()[:5]  # first 5 merges

### `fit_agglomerative()`

Returns `self`. Set either `n_clusters` or `distance_threshold`, not both.

In [ ]:
cluster2.fit_agglomerative(n_clusters=4, linkage="ward")
cluster2.plot_clusters(x="Annual Income", y="Spending Score", show_centers=False)

In [ ]:
cluster2.evaluate()  # no inertia — not a native Agglomerative attribute

In [ ]:
cluster2.get_clustered_data(column_name="Segment").head()

In [ ]:
cluster2.cluster_summary()

### Clustering by distance instead of a fixed count

Pass `distance_threshold` and leave `n_clusters=None`; scikit-learn then
decides how many clusters to form.

In [ ]:
cluster3 = Cluster(data=df, features=["Annual Income", "Spending Score"], scale=True)
cluster3.fit_agglomerative(n_clusters=None, distance_threshold=5, linkage="ward")
cluster3.evaluate()

---
That's every public method on `Cluster`. See the
[README](https://github.com/shashikant-ds636/easycluster) for the full API
reference and design notes.